# Bow 기반 텍스트 분석 연습

1. 적절한 데이터 셋을 찾거나 생성하고, 적절한 전처리를 진행한다. [01_proprocessing.ipynb]
2. TF-IDF Vectorizer를 이용하여 벡터화 한다
3. Cosine Similarity를 걔산하여 입력된 문자열의 긍/ 부정을 판단한다

In [1]:
import pickle

with open("data/preprocessed_tokens.pkl", "rb") as f:
    tokens_all = pickle.load(f)

print(len(tokens_all))
print(tokens_all[:2])

5000
[['함평', '여고생', '사건', '아버지', '친구', '무죄'], ['한국', '한자어', '사전', '텍스트', '변환', '폰트', '서체', '개발', '폰트', '유니코드', '사전', '통합', '편집', '개발', '사업', '진행', '하다', '오다']]


In [2]:
corpus_joined = [" ".join(tokens) for tokens in tokens_all]

In [4]:
!pip install scikit-learn

  Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl (8.0 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   -- ------------------------------------- 2.4/36.5 MB 19.1 MB/s eta 0:00:02
   -------------------- ------------------- 18.6/36.5 MB 51.0 MB/s eta 0:00:01
   ---------------------------------------  36.4/36.5 MB 64.3 MB/s eta 0:00:01
   ---------------------------------------- 36.5/36.5 MB 59.5 MB/s  0:00:00
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ----

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=3000,
    min_df=3,
    max_df=0.8
)

X = vectorizer.fit_transform(corpus_joined)

print(X.shape)

(5000, 3000)


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

def search_similar(query, top_k=5):
    
    # 동일 전처리
    processed = preprocess_korean(query)
    query_vec = vectorizer.transform([" ".join(processed)])
    
    # 코사인 유사도 계산
    similarities = cosine_similarity(query_vec, X)[0]
    
    # 상위 k개 인덱스
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    results = []
    for idx in top_indices:
        results.append((corpus_joined[idx], similarities[idx]))
    
    return results

In [8]:
import re
from konlpy.tag import Okt

okt = Okt()

def preprocess_korean(text):
    text = re.sub(r"[^가-힣\s]", "", text)
    tokens = okt.pos(text, stem=True)
    tokens = [
        word for word, pos in tokens
        if pos in ["Noun","Verb","Adjective"]
        and len(word) > 1
    ]
    return tokens

In [9]:
results = search_similar("한국 경제 성장 문제")

for sentence, score in results:
    print(f"유사도: {score:.4f}")
    print(sentence)
    print("-"*50)

유사도: 0.5142
기후변화 대응 있다 개발도상국 대한 규제 경제 성장 저해 하다 반면 녹색 성장 패러다임 도입 온실가스 배출 환경오염 줄이다 동시 지속 성장 추구 하다 하다 지속가능성 개념 하나로 개발도상국 빈곤 감소 경제 성장 도모 하다 기후변화 대응 하다 있다
--------------------------------------------------
유사도: 0.3086
성장 형평 능률 기조 자력 성장 구조 확립 사회 개발 하다 형평 증진 시키다 기술 혁신 능률 향상 시키다 목표 하다
--------------------------------------------------
유사도: 0.2905
현재 미국 영국 비롯 대다수 선진 경제학자 사이 계획 경제 관료 효율 해결 하다 없다 경제 구조 평가 받다 있다 대부분 국가 경제 정책 채택 되다 않다 경제 원리 남아 있다
--------------------------------------------------
유사도: 0.2890
한국 법무부 법무 미국 법무부 법률자문 실의 기능 하다
--------------------------------------------------
유사도: 0.2876
첫째 빠르다 성장하다 글로벌 수소 경제 한국 앞서다 기술 바탕 계시 선도 하다 있다
--------------------------------------------------


In [10]:
positive_sentences = [
    "정말 재미있다",
    "최고의 영화다",
    "감동적이고 훌륭하다",
    "완전 만족한다",
    "좋은 경험이었다"
]

negative_sentences = [
    "정말 별로다",
    "시간 낭비였다",
    "지루하고 재미없다",
    "최악의 경험이다",
    "다시는 보고 싶지 않다"
]

In [11]:
pos_processed = [" ".join(preprocess_korean(s)) for s in positive_sentences]
neg_processed = [" ".join(preprocess_korean(s)) for s in negative_sentences]

In [12]:
pos_vec = vectorizer.transform(pos_processed)
neg_vec = vectorizer.transform(neg_processed)

import numpy as np

pos_mean = pos_vec.mean(axis=0)
neg_mean = neg_vec.mean(axis=0)

In [15]:
import numpy as np

In [23]:
from sklearn.metrics.pairwise import cosine_similarity

def predict_sentiment(text):
    
    processed = " ".join(preprocess_korean(text))
    vec = vectorizer.transform([processed])
    
    pos_mean = pos_vec.mean(axis=0).A
    neg_mean = neg_vec.mean(axis=0).A
    
    if pos_sim > neg_sim:
        return "긍정", pos_sim, neg_sim
    else:
        return "부정", pos_sim, neg_sim

In [24]:
from sklearn.metrics.pairwise import cosine_similarity

def predict_sentiment(text):
    
    processed = " ".join(preprocess_korean(text))
    vec = vectorizer.transform([processed])
    
    pos_sim = cosine_similarity(vec, pos_mean)[0][0]
    neg_sim = cosine_similarity(vec, neg_mean)[0][0]
    
    if pos_sim > neg_sim:
        return "긍정", pos_sim, neg_sim
    else:
        return "부정", pos_sim, neg_sim

In [25]:
pos_mean = pos_vec.mean(axis=0).A
neg_mean = neg_vec.mean(axis=0).A

In [26]:
predict_sentiment("이 영화는 정말 감동적이다")

('긍정', np.float64(0.3388281219174473), np.float64(0.0))